# Grade Change Intelligence: Paper Making Process Simulator

## 1. Project Context & Problem Statement
This notebook implements a high-fidelity simulator for the paper-making process, specifically designed to generate rich, ML-ready datasets for **Grade Change Intelligence**. It incorporates realistic process dynamics, sensor artifacts, and diverse transition scenarios based on industrial research.

### The Challenge
During grade changes, paper machines often produce "off-spec" paper due to complex, interacting dynamics, transport delays, and disturbances. The goal is to create a simulator that can generate data reflecting these challenges, enabling the development of predictive models to minimize losses and stabilize quality faster.

### Key Enhancements Implemented
1.  **Explicit Setpoints & Target Labels:** `basis_weight_setpoint`, `moisture_setpoint`, `ash_setpoint`, `deviation_percent`, `off_spec` (with grade-specific spec bands).
2.  **Multiple Grade Transitions:** Ability to simulate 40-50 transitions between various grades (A, B, C, etc.).
3.  **Realistic Sensor Behavior:** Includes measurement noise, occasional spikes, calibration drift, and communication glitches.
4.  **Process Disturbances:** Randomly simulates events like steam pressure drops, pump fluctuations, stock consistency variations, and retention loss.
5.  **Strengthened Variable Coupling:** Interconnected dynamics, e.g., Machine Speed affecting Residence Time, which in turn impacts Moisture and Basis Weight.
6.  **Diverse Transition Scenarios:** Simulates various operational scenarios like "Slow Actuator", "Steam Valve Lag", "Aggressive Operator", "Disturbed Transition", and "Recipe Mismatch".

## 2. Simulator Core: `PaperMachineSimulator` Class

### Checkpoint 1: Initialization and Parameters
The `__init__` method sets up the machine's physical parameters, initial states of manipulated and controlled variables, explicit setpoints, and buffers for handling transport delays. It also defines the `recipes` for different paper grades, including their target values and basis weight specification bands.

**Domain Knowledge:** Paper machine dimensions (`width`, `distance_to_scanner`), material properties (`retention`, `consistency`), and typical operating ranges are crucial for realistic simulation. `buffer_size` is set to accommodate the maximum possible transport delay.

**Mathematical/ML Information:** Explicit setpoints and `bw_spec` (basis weight specification) are added to directly support the `deviation_percent` and `off_spec` target labels, which are vital for supervised learning tasks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
import random

class PaperMachineSimulator:
    def __init__(self, dt=1.0):
        self.dt = dt  # Simulation step (seconds)
        self.time = 0.0
        
        # --- Machine Parameters ---
        self.width = 5.0  # Machine width (m)
        self.distance_to_scanner = 100.0  # Distance from wet end to scanner (m)
        self.retention = 0.75  # First-pass retention (initial)
        self.consistency = 0.035  # Stock consistency (initial)
        
        # --- State Initialization ---
        self.current_grade = "Grade_A"
        self.target_grade = "Grade_A"
        self.is_transitioning = False
        self.transition_scenario = "Normal"
        
        # Actuator States (Manipulated Variables)
        self.stock_flow = 1200.0  # GPM
        self.filler_flow = 100.0  # GPM
        self.machine_speed = 800.0  # m/min
        self.steam_pressure = 60.0  # PSI
        
        # Quality States (Controlled Variables)
        self.basis_weight = 60.0  # GSM
        self.moisture = 6.5  # %
        self.ash_content = 10.0  # %
        self.dryer_temp = 120.0  # C
        
        # Setpoints (Explicit)
        self.basis_weight_sp = 60.0
        self.moisture_sp = 6.5
        self.ash_sp = 10.0
        
        # Delay Buffers (for transport delays) - Max steps for 20 min delay at 1s dt
        self.buffer_size = 1200 
        self.stock_buf = deque([self.stock_flow] * self.buffer_size, maxlen=self.buffer_size)
        self.filler_buf = deque([self.filler_flow] * self.buffer_size, maxlen=self.buffer_size)
        self.speed_buf = deque([self.machine_speed] * self.buffer_size, maxlen=self.buffer_size)
        self.steam_buf = deque([self.steam_pressure] * self.buffer_size, maxlen=self.buffer_size)
        
        # Sensor Realism Parameters
        self.bw_calibration_drift = 0.0 # GSM/hour
        self.last_bw_cal_time = 0.0
        self.frozen_sensor_val = None
        self.frozen_sensor_duration = 0
        
        # Recipes (Targets with Spec Bands) - Based on research findings
        self.recipes = {
            "Grade_A": {"bw": 60.0, "speed": 800.0, "steam": 60.0, "filler": 100.0, "moisture": 6.5, "ash": 10.0, "bw_spec": 0.025},
            "Grade_B": {"bw": 85.0, "speed": 720.0, "steam": 85.0, "filler": 150.0, "moisture": 5.8, "ash": 12.0, "bw_spec": 0.025},
            "Grade_C": {"bw": 120.0, "speed": 600.0, "steam": 110.0, "filler": 200.0, "moisture": 5.0, "ash": 15.0, "bw_spec": 0.025}
        }
        
        self.data_log = []

    def get_transport_delay_steps(self, speed):
        # Technical: Delay (s) = Distance (m) / Speed (m/min) * 60
        # Domain: Speed directly impacts the time it takes for material to travel.
        speed_mps = speed / 60.0
        if speed_mps == 0: return self.buffer_size # Prevent division by zero, return max delay
        delay_seconds = self.distance_to_scanner / speed_mps
        return int(delay_seconds / self.dt)

### Checkpoint 2: Sensor Realism and Disturbances
This section introduces functions to simulate realistic sensor behavior and process disturbances, crucial for generating a robust dataset for ML models.

**Domain Knowledge:** Real-world sensors are imperfect. Noise, spikes, and drift are common. Process disturbances like consistency swings or pump fluctuations are inherent to industrial operations and are major causes of off-spec production. Simulating these based on research-backed magnitudes makes the dataset more representative of reality.

**Mathematical/ML Information:** Adding these artifacts helps train ML models to be robust to noisy data and to identify patterns that lead to off-spec conditions even under challenging circumstances. `off_spec` events caused by disturbances are key training examples.

In [ ]:
    def apply_sensor_noise_and_artifacts(self, value, nominal_value, var_name):
        # Measurement Noise (Gaussian white noise, ~0.05% of nominal value)
        noise_std = 0.0005 * nominal_value
        value += np.random.normal(0, noise_std)
        
        # Spikes (Occasional large outliers, 5 sigma, 0.1% chance)
        if random.random() < 0.001: 
            value += np.random.normal(0, 5 * noise_std) * 10 
            
        # Calibration Drift (for Basis Weight only: 0.02 GSM per hour, reset every 8 hours)
        if var_name == "basis_weight":
            drift_rate_per_sec = 0.02 / 3600.0 
            self.bw_calibration_drift += drift_rate_per_sec * self.dt
            value += self.bw_calibration_drift
            if (self.time - self.last_bw_cal_time) > (8 * 3600):
                self.bw_calibration_drift = 0.0
                self.last_bw_cal_time = self.time
                
        # Communication Glitches (Frozen Value: 0.005% chance to freeze for 1-3 steps)
        if self.frozen_sensor_duration > 0:
            self.frozen_sensor_duration -= 1
            return self.frozen_sensor_val
        elif random.random() < 0.00005: 
            self.frozen_sensor_val = value
            self.frozen_sensor_duration = random.randint(1, 3) 
            return self.frozen_sensor_val
            
        return value

    def apply_process_disturbances(self):
        # Consistency Swing: +/- 0.08% over 2-5 mins (periodic, low prob to start)
        if random.random() < 0.0001: 
            swing_magnitude = random.uniform(-0.0008, 0.0008) 
            self.consistency += swing_magnitude
            
        # Retention Loss: -10% drop over 1-3 mins (very low prob)
        if random.random() < 0.00005: 
            self.retention = max(0.65, self.retention - 0.10) 
            
        # Steam Pressure Drop: -8 PSI over 30-60s (extremely rare)
        if random.random() < 0.00002: 
            self.steam_pressure = max(30.0, self.steam_pressure - 8.0)
            
        # Pump Fluctuation: 1.5% oscillation in stock flow (continuous)
        self.stock_flow += np.sin(self.time * 0.1) * (0.015 * self.stock_flow)

PaperMachineSimulator.apply_sensor_noise_and_artifacts = apply_sensor_noise_and_artifacts
PaperMachineSimulator.apply_process_disturbances = apply_process_disturbances

### Checkpoint 3: Step Dynamics and Variable Coupling
The `step` method advances the simulation by one `dt`. It applies process disturbances, updates actuator positions based on target setpoints (with scenario-specific time constants), and then calculates the new state of the controlled variables using mass balance and FOPDT models. Crucially, it strengthens variable coupling.

**Technical:** Actuator dynamics are modeled as first-order lags. Transport delays are dynamically calculated based on machine speed. FOPDT models are used for moisture and ash, incorporating time constants and dead times derived from research.

**Domain Knowledge:** The interdependencies are explicitly modeled: Machine Speed affects Residence Time, which directly influences Moisture. Steam Pressure also impacts Moisture. Basis Weight is a function of delayed Stock Flow and Machine Speed. This interconnectedness is vital for realistic grade change simulation.

**Mathematical/ML Information:** This method also calculates `deviation_percent` and `off_spec` labels based on explicit setpoints and spec bands, making the generated data directly usable for classification and regression tasks in ML.

In [ ]:
    def step(self, target_setpoints=None, transition_info=None):
        # Apply process disturbances before actuator changes
        self.apply_process_disturbances()
        
        if target_setpoints:
            # Coordinated Ramping (Actuator Dynamics) - Time constants from research
            stock_tc = 15.0 # seconds
            speed_tc = 10.0
            steam_tc = 90.0
            filler_tc = 20.0
            
            # Scenario-specific actuator behavior
            if self.transition_scenario == "Slow Actuator":
                stock_tc *= 3 # Stock valve time constant increased by 3x
            elif self.transition_scenario == "Aggressive Operator":
                stock_tc /= 1.5 # Faster ramp
                speed_tc /= 1.5
                steam_tc /= 1.5
            
            self.stock_flow += (target_setpoints["stock_flow"] - self.stock_flow) * (self.dt / stock_tc)
            self.filler_flow += (target_setpoints["filler_flow"] - self.filler_flow) * (self.dt / filler_tc)
            self.machine_speed += (target_setpoints["machine_speed"] - self.machine_speed) * (self.dt / speed_tc)
            
            # Steam pressure with potential lag for "Steam Lag" scenario
            steam_target = target_setpoints["steam_pressure"]
            if self.transition_scenario == "Steam Lag":
                # Simplified additional lag for steam pressure response (45s from research)
                delayed_steam_target = self.steam_buf[max(0, len(self.steam_buf) - int(45/self.dt))]
                self.steam_pressure += (delayed_steam_target - self.steam_pressure) * (self.dt / steam_tc)
            else:
                self.steam_pressure += (steam_target - self.steam_pressure) * (self.dt / steam_tc)

            # Update explicit setpoints for logging
            self.basis_weight_sp = target_setpoints["basis_weight_sp"]
            self.moisture_sp = target_setpoints["moisture_sp"]
            self.ash_sp = target_setpoints["ash_sp"]

        # Update buffers for transport delays
        self.stock_buf.append(self.stock_flow)
        self.filler_buf.append(self.filler_flow)
        self.speed_buf.append(self.machine_speed)
        self.steam_buf.append(self.steam_pressure)
        
        # Calculate Dead Time steps (speed dependent for all variables)
        delay_steps_bw = self.get_transport_delay_steps(self.machine_speed)
        delay_steps_moisture = self.get_transport_delay_steps(self.machine_speed) 
        delay_steps_ash = self.get_transport_delay_steps(self.machine_speed)
        
        # Retrieve delayed inputs from buffers
        delayed_stock = self.stock_buf[max(0, len(self.stock_buf) - delay_steps_bw)]
        delayed_speed_bw = self.speed_buf[max(0, len(self.speed_buf) - delay_steps_bw)]
        delayed_filler = self.filler_buf[max(0, len(self.filler_buf) - delay_steps_ash)]
        delayed_speed_moisture = self.speed_buf[max(0, len(self.speed_buf) - delay_steps_moisture)]
        delayed_steam = self.steam_buf[max(0, len(self.steam_buf) - delay_steps_moisture)]
        
        # --- Process Dynamics (Strengthened Coupling) ---
        
        # 1. Basis Weight (Mass Balance) - Coupled with Machine Speed
        if delayed_speed_bw > 0:
            bw_raw = (delayed_stock * self.consistency * self.retention * 1000) / (delayed_speed_bw * self.width)
            self.basis_weight += (bw_raw - self.basis_weight) * (self.dt / 15.0) # BW time constant 15s
        
        # 2. Moisture (FOPDT) - Coupled with Machine Speed (Residence Time) and Steam Pressure
        residence_time_factor = 1.0 / (delayed_speed_moisture / 60.0) # Inverse of speed in m/s
        steam_effect = delayed_steam / 20.0 # Simplified gain for steam effect
        
        moisture_target = 5.0 + (residence_time_factor * 0.1) - steam_effect
        self.moisture += (moisture_target - self.moisture) * (self.dt / 90.0) # Moisture time constant 90s
        
        # 3. Ash Content (FOPDT) - Coupled with Filler Flow and Stock Flow
        if delayed_stock > 0:
            ash_target = (delayed_filler / delayed_stock) * 100.0
            self.ash_content += (ash_target - self.ash_content) * (self.dt / 30.0) # Ash time constant 30s
        
        # 4. Dryer Temp (Directly related to Steam Pressure)
        temp_target = 100.0 + (self.steam_pressure * 0.8)
        self.dryer_temp += (temp_target - self.dryer_temp) * (self.dt / 5.0) # Fast response
        
        # --- Sensor Realism & Observed Values ---
        bw_obs = self.apply_sensor_noise_and_artifacts(self.basis_weight, self.recipes[self.current_grade]["bw"] if self.current_grade in self.recipes else self.basis_weight_sp, "basis_weight")
        moisture_obs = self.apply_sensor_noise_and_artifacts(self.moisture, self.recipes[self.current_grade]["moisture"] if self.current_grade in self.recipes else self.moisture_sp, "moisture")
        ash_obs = self.apply_sensor_noise_and_artifacts(self.ash_content, self.recipes[self.current_grade]["ash"] if self.current_grade in self.recipes else self.ash_sp, "ash_content")
        
        # --- Target Labels ---
        deviation_percent = abs(bw_obs - self.basis_weight_sp) / self.basis_weight_sp * 100 if self.basis_weight_sp > 0 else 0
        off_spec = 1 if deviation_percent > self.recipes[self.current_grade]["bw_spec"] * 100 else 0
        
        # Logging
        self.data_log.append({
            "timestamp": self.time,
            "basis_weight": bw_obs,
            "stock_flow": self.stock_flow,
            "filler_flow": self.filler_flow,
            "machine_speed": self.machine_speed,
            "steam_pressure": self.steam_pressure,
            "moisture": moisture_obs,
            "ash_content": ash_obs,
            "dryer_temp": self.dryer_temp,
            "basis_weight_setpoint": self.basis_weight_sp,
            "moisture_setpoint": self.moisture_sp,
            "ash_setpoint": self.ash_sp,
            "deviation_percent": deviation_percent,
            "off_spec": off_spec,
            "grade": self.current_grade,
            "is_transitioning": transition_info["is_transitioning"] if transition_info else False,
            "transition_scenario": transition_info["scenario"] if transition_info else "SteadyState"
        })
        
        self.time += self.dt

    def run_transition(self, start_grade, end_grade, duration_steps=1200, scenario="Normal"):
        self.current_grade = start_grade
        self.target_grade = end_grade
        self.is_transitioning = True
        self.transition_scenario = scenario
        
        start_vals = self.recipes[start_grade]
        end_vals = self.recipes[end_grade]
        
        self.basis_weight_sp = start_vals["bw"]
        self.moisture_sp = start_vals["moisture"]
        self.ash_sp = start_vals["ash"]

        stock_flow_calc_end = (end_vals["bw"] * end_vals["speed"] * self.width) / (self.consistency * self.retention * 1000)
        stock_ramp_rate = (stock_flow_calc_end - self.stock_flow) / duration_steps
        speed_ramp_rate = (end_vals["speed"] - self.machine_speed) / duration_steps
        steam_ramp_rate = (end_vals["steam"] - self.steam_pressure) / duration_steps
        filler_ramp_rate = (end_vals["filler"] - self.filler_flow) / duration_steps

        bw_sp_ramp_rate = (end_vals["bw"] - self.basis_weight_sp) / duration_steps
        moisture_sp_ramp_rate = (end_vals["moisture"] - self.moisture_sp) / duration_steps
        ash_sp_ramp_rate = (end_vals["ash"] - self.ash_sp) / duration_steps

        for i in range(duration_steps):
            current_stock_target = self.stock_flow + stock_ramp_rate
            current_speed_target = self.machine_speed + speed_ramp_rate
            current_steam_target = self.steam_pressure + steam_ramp_rate
            current_filler_target = self.filler_flow + filler_ramp_rate

            current_bw_sp = self.basis_weight_sp + bw_sp_ramp_rate
            current_moisture_sp = self.moisture_sp + moisture_sp_ramp_rate
            current_ash_sp = self.ash_sp + ash_sp_ramp_rate

            if scenario == "Disturbed Trans." and i == int(duration_steps / 2):
                self.consistency += 0.0015
            elif scenario == "Recipe Mismatch":
                current_stock_target *= 1.02

            target_setpoints = {
                "stock_flow": current_stock_target,
                "filler_flow": current_filler_target,
                "machine_speed": current_speed_target,
                "steam_pressure": current_steam_target,
                "basis_weight_sp": current_bw_sp,
                "moisture_sp": current_moisture_sp,
                "ash_sp": current_ash_sp
            }
            self.step(target_setpoints, {"is_transitioning": True, "scenario": scenario})
            
        self.is_transitioning = False
        self.current_grade = end_grade
        self.transition_scenario = "SteadyState"

    def run_steady_state(self, grade, duration_steps):
        self.current_grade = grade
        self.is_transitioning = False
        self.transition_scenario = "SteadyState"
        
        grade_vals = self.recipes[grade]
        self.basis_weight_sp = grade_vals["bw"]
        self.moisture_sp = grade_vals["moisture"]
        self.ash_sp = grade_vals["ash"]

        stock_flow_calc = (grade_vals["bw"] * grade_vals["speed"] * self.width) / (self.consistency * self.retention * 1000)
        self.recipes[grade]["stock_flow_calc"] = stock_flow_calc

        target_setpoints = {
            "stock_flow": stock_flow_calc,
            "filler_flow": grade_vals["filler"],
            "machine_speed": grade_vals["speed"],
            "steam_pressure": grade_vals["steam"],
            "basis_weight_sp": grade_vals["bw"],
            "moisture_sp": grade_vals["moisture"],
            "ash_sp": grade_vals["ash"]
        }

        for _ in range(duration_steps):
            self.step(target_setpoints, {"is_transitioning": False, "scenario": "SteadyState"})

    def get_dataset(self):
        return pd.DataFrame(self.data_log)

PaperMachineSimulator.step = step
PaperMachineSimulator.run_transition = run_transition
PaperMachineSimulator.run_steady_state = run_steady_state
PaperMachineSimulator.get_dataset = get_dataset

### Checkpoint 4: Generating Diverse Scenarios
This section orchestrates the simulation, generating a large dataset with multiple grade transitions and varied scenarios. This is crucial for training robust ML models that can predict off-spec conditions under different operational challenges.

**Technical:** The simulation loop iterates through a predefined number of transitions, randomly selecting start and end grades, and a transition scenario. Each transition includes a steady-state period before and after the ramp.

**Domain Knowledge:** Generating 40-50 transitions with different scenarios (Normal, Slow Actuator, Disturbed, etc.) mimics the real-world operational variability. This allows the ML model to learn from both successful and challenging grade changes.

**ML Information:** The resulting dataset will contain `off_spec` labels and `deviation_percent` for each timestamp, making it ideal for supervised learning tasks like classification (predicting `off_spec`) and regression (predicting `deviation_percent`).

In [ ]:
# --- Execution --- 
sim = PaperMachineSimulator()

# Pre-calculate stock_flow_calc for all recipes
for grade_name, grade_data in sim.recipes.items():
    stock_flow_calc = (grade_data["bw"] * grade_data["speed"] * sim.width) / (sim.consistency * sim.retention * 1000)
    sim.recipes[grade_name]["stock_flow_calc"] = stock_flow_calc

grades = list(sim.recipes.keys())
transition_scenarios = ["Normal", "Slow Actuator", "Steam Lag", "Aggressive Operator", "Disturbed Trans.", "Recipe Mismatch"]

num_transitions = 50 
steady_state_duration = 600 
transition_duration = 600 

for i in range(num_transitions):
    start_grade = random.choice(grades)
    end_grade = random.choice([g for g in grades if g != start_grade])
    scenario = random.choice(transition_scenarios)
    
    print(f"Simulating transition {i+1}/{num_transitions}: {start_grade} -> {end_grade} (Scenario: {scenario})")
    sim.run_steady_state(start_grade, duration_steps=steady_state_duration)
    sim.run_transition(start_grade, end_grade, duration_steps=transition_duration, scenario=scenario)
    sim.run_steady_state(end_grade, duration_steps=steady_state_duration)

df = sim.get_dataset()
df.to_csv('paper_making_dataset_enhanced.csv', index=False)
print("Enhanced dataset generated successfully: paper_making_dataset_enhanced.csv")

## 5. Visualizing the Enhanced Trajectories
This section provides visualizations of the generated data, highlighting the new features like explicit setpoints, spec bands, and off-spec events.

In [ ]:
plt.figure(figsize=(15, 12))

plt.subplot(4, 1, 1)
plt.plot(df["timestamp"], df["basis_weight"], label='Basis Weight (GSM)', color='blue', alpha=0.7)
plt.plot(df["timestamp"], df["basis_weight_setpoint"], label='BW Setpoint', color='red', linestyle='--')
bw_lower_spec = df.apply(lambda row: row["basis_weight_setpoint"] * (1 - sim.recipes[row["grade"]]["bw_spec"]), axis=1)
bw_upper_spec = df.apply(lambda row: row["basis_weight_setpoint"] * (1 + sim.recipes[row["grade"]]["bw_spec"]), axis=1)
plt.fill_between(df["timestamp"], bw_lower_spec, bw_upper_spec, color='red', alpha=0.1, label='BW Spec Band')
plt.scatter(df[df["off_spec"] == 1]["timestamp"], df[df["off_spec"] == 1]["basis_weight"], color='red', marker='x', s=50, label='Off-Spec Event', zorder=5)
plt.ylabel('Basis Weight (GSM)')
plt.legend(loc='upper right')
plt.title('Enhanced Paper Machine Simulation: Basis Weight Trajectory')
plt.grid(True)

plt.subplot(4, 1, 2)
plt.plot(df["timestamp"], df["moisture"], label='Moisture (%)', color='green', alpha=0.7)
plt.plot(df["timestamp"], df["moisture_setpoint"], label='Moisture Setpoint', color='purple', linestyle='--')
plt.ylabel('Moisture (%)')
plt.legend(loc='upper right')
plt.grid(True)

plt.subplot(4, 1, 3)
plt.plot(df["timestamp"], df["ash_content"], label='Ash Content (%)', color='brown', alpha=0.7)
plt.plot(df["timestamp"], df["ash_setpoint"], label='Ash Setpoint', color='orange', linestyle='--')
plt.ylabel('Ash Content (%)')
plt.legend(loc='upper right')
plt.grid(True)

plt.subplot(4, 1, 4)
plt.plot(df["timestamp"], df["stock_flow"], label='Stock Flow (GPM)', color='cyan', alpha=0.7)
plt.plot(df["timestamp"], df["machine_speed"], label='Machine Speed (m/min)', color='magenta', alpha=0.7)
plt.ylabel('Flow / Speed')
plt.xlabel('Time (s)')
plt.legend(loc='upper right')
plt.grid(True)

plt.tight_layout()
plt.show()